# Museum Art Guide - run every experiment on Google Colab

1. `Runtime > Change runtime type > T4 GPU` (the GPU speeds up the ResNet fine-tuning; everything else runs on CPU too).
2. Replace `<GITHUB_USER>` below with the owner of the repository, then run the cells in order.
3. At the end, download `results.zip` and commit the `results/` folder **from your own computer** (see the last cell).

The WikiArt images stay on Colab / your Drive: they are never committed (licence unknown).

In [ ]:
!nvidia-smi -L || echo "no GPU: fine for steps 0-3, slower for step 4"

In [ ]:
REPO = "https://github.com/<GITHUB_USER>/museum-art-guide.git"
!git clone -q $REPO museum-art-guide
%cd museum-art-guide
!pip install -q -r requirements.txt && pip install -q -e .
!python -m pytest -q

## Data (optional: keep it on Google Drive so that a new Colab session does not download 4 GB again)

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/museum-art-guide-data/wikiart_style5
    !ln -sfn /content/drive/MyDrive/museum-art-guide-data/wikiart_style5 data/wikiart_style5


In [ ]:
# Plan (dry run) then download: ~4 GB streamed, a few minutes on Colab. Re-running resumes.
!bash scripts/build_dataset.sh
!cat data/wikiart_style5/subset_info.json

## Step 0 - exploration and artist-grouped split

In [ ]:
!python scripts/00_explore.py
from IPython.display import Image, display
display(Image("results/00_explore/samples.png"))

## Step 1 - classical baseline (logistic regression)

In [ ]:
!python scripts/01_baseline.py
!python scripts/01_baseline.py --split random   # to measure artist leakage

## Step 2 - MLP from scratch (sweep on validation, 3 seeds)

In [ ]:
!python scripts/02_train_mlp.py --features pixels32
!python scripts/02_train_mlp.py --features handcrafted
display(Image("results/02_mlp/pixels32/training_history.png"))

## Step 3 - failure diagnosis

In [ ]:
!python scripts/03_diagnose_mlp.py
display(Image("results/03_diagnosis/diagnosis_overview.png"))
!cat results/03_diagnosis/summary.md

## Step 4 - pretrained ResNet-18 (frozen and fine-tuned)

In [ ]:
!python scripts/04_train_resnet.py --mode both
display(Image("results/04_resnet/finetune/training_history.png"))
display(Image("results/04_resnet/finetune/errors.png"))

## Step 5 - CPU benchmark
Colab's CPU is not a kiosk: **also run this step on one of your laptops** (`python scripts/05_benchmark_cpu.py`)
and report the hardware. Latency does not depend on where the model was trained.

In [ ]:
!CUDA_VISIBLE_DEVICES="" python scripts/05_benchmark_cpu.py

## Step 6 - summary for the director

In [ ]:
!python scripts/06_summary.py
display(Image("results/tradeoff.png"))

In [ ]:
# Download the results (JSON + figures, a few MB). samples.png / errors.png show paintings:
# use them in the slides, but they are git-ignored (not ours to publish).
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")
# Then on your computer: unzip into the repo, `git add results && git commit -m "results: ..."`, push.